# Transcriptomic Classification of Breast Cancer Molecular Subtypes

**Supervised classification, external validation and unsupervised structure analysis of gene-expression profiles from the TCGA breast cancer cohort (TCGA-BRCA)**

*Models are trained on the full QC-filtered transcriptome (no differential-expression pre-selection).*

## Abstract

This notebook investigates whether the molecular subtype of a breast tumour (Luminal A, Luminal B or Basal-like) can be recovered from bulk RNA-seq gene-expression profiles. Five supervised classifiers — *k*-nearest neighbours (KNN), random forest (RF), logistic regression (LR), gradient-boosted trees (XGBoost) and support vector machines (SVM) — are tuned by cross-validated hyperparameter search on a predefined training partition and are evaluated on a held-out test partition and by repeated stratified cross-validation. The tuned models are subsequently applied to an independent external cohort (INC Bogotá; Luminal B samples only). Finally, the intrinsic structure of the data is explored without label information using principal component analysis (PCA), UMAP, *k*-means and hierarchical clustering.

## Contents

1. Environment and configuration
2. Data loading and preprocessing
3. Training/test partition
4. Supervised classification
5. External validation on the INC Bogotá cohort
6. Unsupervised analysis
7. Summary and limitations

## Input data

The notebook expects the following files in the `data/` directory (paths are configurable in Section 1):

| File | Content |
|---|---|
| `qc_filtered_transcriptome.tsv` | QC-filtered TCGA-BRCA expression matrix in log2(FPKM + 1) units, stored as genes × samples. Two metadata rows (subtype and patient identifier) precede the gene rows. |
| `train_test_split.csv` | Predefined partition of the patients. Columns: `patient_id` and `split` (`train` or `test`). |
| `inc_luminal_b_cohort.csv` | External cohort (INC Bogotá): VST-normalised expression matrix of Luminal B samples, stored as genes × samples. |

## Outputs

Every output file carries the prefix `rna_filtered_` to avoid name collisions with other modules of the repository. Figures (300 dpi PNG) are written to `figures/`, summary tables (CSV) to `results/`, and the gene lists intended for functional-enrichment analysis (DAVID) to `results/`.

## Reproducibility

All stochastic components use a fixed seed (`RANDOM_STATE = 42`). The versions of the main libraries are printed at the end of Section 1, and the reference environment is specified in `requirements_rna.txt`.

## 1. Environment and configuration

Imports, file paths, the random seed, the class encoding and the colour palette are declared once in this section so that the remainder of the notebook contains no hard-coded constants.

In [ ]:
import platform
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
import umap
import xgboost
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.stats import gaussian_kde, randint, uniform, wilcoxon
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             adjusted_rand_score, classification_report,
                             confusion_matrix, f1_score, silhouette_score)
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV,
                                     RepeatedStratifiedKFold, StratifiedKFold,
                                     cross_validate)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

# Suppress only known, benign warnings: the deprecation of `multi_class` in
# LogisticRegression (scikit-learn >= 1.5) and non-convergence notices of the
# SAGA solver raised during the grid search.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [ ]:
# ----------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------
RANDOM_STATE = 42

DATA_DIR = Path("data")            # adjust to the location of the input files

# Input files
TCGA_EXPRESSION_FILE = DATA_DIR / "qc_filtered_transcriptome.tsv"
SPLIT_FILE = DATA_DIR / "train_test_split.csv"
INC_EXPRESSION_FILE = DATA_DIR / "inc_luminal_b_cohort.csv"

# Output directories and file prefix
FIGURES_DIR = Path("figures")
TABLES_DIR = Path("results")
DAVID_DIR = Path("results")
OUTPUT_PREFIX = "rna_filtered"
for directory in (FIGURES_DIR, TABLES_DIR, DAVID_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Class encoding (fixes the class order in every table and figure)
LABEL_TO_CODE = {"LumA": 0, "LumB": 1, "Basal": 2}
CODE_TO_LABEL = {code: label for label, code in LABEL_TO_CODE.items()}
CLASS_NAMES = [CODE_TO_LABEL[c] for c in sorted(CODE_TO_LABEL)]
COLOR_MAP = {"LumA": "#2196F3", "LumB": "#FF9800", "Basal": "#F44336"}

# Genes removed from the KNN feature space (flagged for missing values during
# preliminary data screening; distance-based methods require complete input).
KNN_EXCLUDED_GENES = ["RAB27B"]

plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300, "savefig.bbox": "tight"})


def save_figure(name):
    """Save the current matplotlib figure to ``FIGURES_DIR`` as a PNG file."""
    plt.savefig(FIGURES_DIR / f"{OUTPUT_PREFIX}_{name}.png")


def save_table(table, name, directory=TABLES_DIR):
    """Save a DataFrame or Series as CSV, using the module-specific file prefix."""
    table.to_csv(directory / f"{OUTPUT_PREFIX}_{name}.csv")

In [ ]:
print(f"Python {platform.python_version()} | NumPy {np.__version__} | pandas {pd.__version__} | "
      f"SciPy {scipy.__version__} | scikit-learn {sklearn.__version__} | "
      f"XGBoost {xgboost.__version__} | umap-learn {umap.__version__}")

## 2. Data loading and preprocessing

### 2.1 Loading the expression matrix

The source table is stored in a genes × samples layout. Its first column (`Gen`) contains feature names; the first two rows contain sample-level metadata (subtype and patient identifier) and are followed by one row per gene. Sample columns follow the naming convention `BRCA_<subtype>[.<n>]`, where the numeric suffix is appended by pandas to disambiguate repeated column names.

In [ ]:
expression_raw = pd.read_csv(TCGA_EXPRESSION_FILE, sep="\t")

# The two metadata rows are labelled "Subtipo" and "IDPaciente" in the source file.
assert list(expression_raw.iloc[:2, 0]) == ["Subtipo", "IDPaciente"], \
    "Unexpected layout: the first two rows must hold the subtype and patient-identifier metadata."

print(f"Raw table: {expression_raw.shape[0]:,} rows x {expression_raw.shape[1]:,} columns")
expression_raw.iloc[:5, :5]

### 2.2 Reshaping to a samples × genes matrix

The matrix is transposed so that each row represents one sample and each column one gene. The patient identifier is retained as a metadata column, and the molecular subtype is parsed from the sample identifier (the redundant subtype metadata row is discarded).

In [ ]:
df = (expression_raw
      .rename(columns={"Gen": "Feature"})
      .drop(index=0)                      # subtype row: redundant with the sample identifier
      .reset_index(drop=True))

samples = df.set_index("Feature").T
samples.index.name = "SampleID"
samples.columns.name = None
samples = samples.reset_index().rename(columns={"IDPaciente": "PatientID"})

# Subtype label parsed from the sample identifier, e.g. "BRCA_LumA.3" -> "LumA"
samples["Subtype"] = samples["SampleID"].str.extract(r"BRCA_([A-Za-z]+)", expand=False)

print(f"Samples x columns: {samples.shape}")
samples[["SampleID", "PatientID", "Subtype"]].head()

### 2.3 Label encoding

Subtype labels are mapped to integers with an explicit dictionary (LumA = 0, LumB = 1, Basal = 2). The explicit mapping fixes the class order across all models, tables and figures.

In [ ]:
samples["Subtype_Encoded"] = samples["Subtype"].map(LABEL_TO_CODE)
assert samples["Subtype_Encoded"].notna().all(), "Unrecognised subtype label(s) found."

print("Class distribution (all samples):")
print(samples["Subtype"].value_counts().reindex(CLASS_NAMES).to_string())

## 3. Training/test partition

The partition is predefined (`train_test_split.csv`) and is merged with the sample table using the patient identifier. A fixed, externally defined partition ensures that every model is evaluated on exactly the same held-out samples.

In [ ]:
split_table = pd.read_csv(SPLIT_FILE)

merged = samples.merge(split_table[["patient_id", "split"]],
                       left_on="PatientID", right_on="patient_id", how="left")

# Fail fast if the partition is incomplete or ambiguous
assert len(merged) == len(samples), "Duplicated patient identifiers in the split file."
assert merged["split"].notna().all(), f"{merged['split'].isna().sum()} sample(s) without a split assignment."
assert set(merged["split"]) <= {"train", "test"}, "The split column must contain only 'train' and 'test'."

pd.crosstab(merged["split"], merged["Subtype"], margins=True).reindex(columns=CLASS_NAMES + ["All"])

In [ ]:
METADATA_COLUMNS = ["SampleID", "PatientID", "Subtype", "Subtype_Encoded", "patient_id", "split"]

X = merged.drop(columns=METADATA_COLUMNS).astype(float)     # samples x genes
y = merged["Subtype_Encoded"]

train_mask = (merged["split"] == "train").to_numpy()
test_mask = (merged["split"] == "test").to_numpy()

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train: {y_train.shape} | y_test: {y_test.shape}")
print(f"Missing values in the expression matrix: {int(X.isna().sum().sum())}")

## 4. Supervised classification

### 4.1 Model-selection strategy

Hyperparameters are selected exclusively on the training partition by 5-fold stratified cross-validation (`cv_tuning`). The held-out test partition is used only for the final assessment of each tuned model. Univariate feature selection (ANOVA F-test, `SelectKBest`) and feature scaling are embedded in scikit-learn pipelines so that both are re-fitted within every training fold, which prevents information leakage from the validation folds.

KNN, RF, LR and SVM are tuned with accuracy as the selection criterion; XGBoost is tuned with macro-averaged F1 in combination with class-balanced sample weights. A complementary comparison of all tuned models on the complete data set is provided in Section 4.8 (`cv_report`).

In [ ]:
cv_tuning = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)                 # hyperparameter search
cv_report = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)          # final model comparison

### 4.2 Evaluation helper

The helper below reports training and test accuracy, the per-class classification report and the confusion matrix on the test partition, and stores the summary metrics of each model for the comparison table in Section 4.7.

In [ ]:
holdout_results = {}


def evaluate_on_holdout(name, model, X_tr, y_tr, X_te, y_te):
    """Evaluate a fitted classifier on the training and test partitions.

    Prints the accuracies and the per-class classification report, plots the
    confusion matrix of the test partition and stores the summary metrics in
    ``holdout_results``.
    """
    labels = sorted(CODE_TO_LABEL)
    y_pred = model.predict(X_te)
    train_acc = accuracy_score(y_tr, model.predict(X_tr))
    test_acc = accuracy_score(y_te, y_pred)
    holdout_results[name] = {
        "Train accuracy": train_acc,
        "Test accuracy": test_acc,
        "Test macro-F1": f1_score(y_te, y_pred, average="macro"),
    }

    print(f"{name}: training accuracy = {train_acc:.4f} | test accuracy = {test_acc:.4f}\n")
    print(classification_report(y_te, y_pred, labels=labels, target_names=CLASS_NAMES, digits=2))

    cm = confusion_matrix(y_te, y_pred, labels=labels)
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES).plot(cmap="Blues", values_format="d")
    plt.title(f"Confusion matrix – {name} (test set)")
    save_figure(f"confusion_matrix_{name.lower().replace(' ', '_')}")
    plt.show()

### 4.3 Model 1: k-nearest neighbours (KNN)

Univariate feature selection, standardisation and a KNN classifier are chained in a pipeline. The number of selected genes (`k`), the number of neighbours, the weighting scheme and the distance metric are tuned by exhaustive grid search (84 configurations × 5 folds). Because distance-based methods require a complete input matrix, the genes listed in `KNN_EXCLUDED_GENES` are removed from the KNN feature space.

In [ ]:
X_train_knn = X_train.drop(columns=KNN_EXCLUDED_GENES)
X_test_knn = X_test.drop(columns=KNN_EXCLUDED_GENES)
X_knn = X.drop(columns=KNN_EXCLUDED_GENES)

print(f"KNN feature space: {X_train_knn.shape[1]:,} genes (excluded: {', '.join(KNN_EXCLUDED_GENES)})")
print(f"Missing values in the KNN training matrix: {int(X_train_knn.isna().sum().sum())}")

In [ ]:
knn_pipeline = Pipeline([
    ("sel", SelectKBest(f_classif)),
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(algorithm="brute")),
])

knn_param_grid = {
    "sel__k": [200, 500, 1000],
    "knn__n_neighbors": [3, 5, 7, 9, 11, 15, 21],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "cosine"],
}

knn_search = GridSearchCV(knn_pipeline, param_grid=knn_param_grid, cv=cv_tuning,
                          scoring="accuracy", n_jobs=-1, verbose=1)
knn_search.fit(X_train_knn, y_train)
knn_best = knn_search.best_estimator_

print("Best hyperparameters:", knn_search.best_params_)
print(f"Best cross-validated accuracy: {knn_search.best_score_:.4f}")

In [ ]:
evaluate_on_holdout("KNN", knn_best, X_train_knn, y_train, X_test_knn, y_test)

### 4.4 Model 2: Random forest (RF)

Hyperparameters are explored by randomised search (50 configurations × 5 folds). The fraction of genes considered at each split (`max_features`) is restricted to the range 2–18 %.

In [ ]:
rf_base = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

rf_param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 30, 40, 50],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "max_features": [0.02, 0.05, 0.1, 0.15, 0.18],      # fraction of features per split
    "criterion": ["gini", "entropy"],
}

rf_search = RandomizedSearchCV(rf_base, param_distributions=rf_param_dist, n_iter=50,
                               scoring="accuracy", cv=cv_tuning,
                               random_state=RANDOM_STATE, n_jobs=-1, verbose=0)
rf_search.fit(X_train, y_train)
rf_best = rf_search.best_estimator_

print("Best hyperparameters:", rf_search.best_params_)
print(f"Best cross-validated accuracy: {rf_search.best_score_:.4f}")

In [ ]:
evaluate_on_holdout("Random Forest", rf_best, X_train, y_train, X_test, y_test)

### 4.5 Model 3: Logistic regression (LR)

A standardisation step precedes an L1- or L2-penalised logistic regression. The grid is specified as a list of dictionaries so that only solver / penalty / multi-class combinations supported by scikit-learn are evaluated (40 configurations × 5 folds).

In [ ]:
logreg_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("logisticregression", LogisticRegression(random_state=RANDOM_STATE)),
])

logreg_param_grid = [
    {   # liblinear: one-vs-rest only; supports L1 and L2
        "logisticregression__C": [0.001, 0.01, 0.1, 1],
        "logisticregression__penalty": ["l1", "l2"],
        "logisticregression__solver": ["liblinear"],
        "logisticregression__multi_class": ["ovr"],
    },
    {   # lbfgs / newton-cg: L2 only
        "logisticregression__C": [0.001, 0.01, 0.1, 1],
        "logisticregression__penalty": ["l2"],
        "logisticregression__solver": ["lbfgs", "newton-cg"],
        "logisticregression__multi_class": ["ovr", "multinomial"],
    },
    {   # saga: supports L1 and L2
        "logisticregression__C": [0.001, 0.01, 0.1, 1],
        "logisticregression__penalty": ["l1", "l2"],
        "logisticregression__solver": ["saga"],
        "logisticregression__multi_class": ["ovr", "multinomial"],
    },
]

logreg_search = GridSearchCV(logreg_pipeline, param_grid=logreg_param_grid, cv=cv_tuning,
                             scoring="accuracy", n_jobs=-1, verbose=1)
logreg_search.fit(X_train, y_train)
logreg_best = logreg_search.best_estimator_

print("Best hyperparameters:", logreg_search.best_params_)
print(f"Best cross-validated accuracy: {logreg_search.best_score_:.4f}")

In [ ]:
evaluate_on_holdout("Logistic Regression", logreg_best, X_train, y_train, X_test, y_test)

### 4.6 Model 4: Gradient-boosted trees (XGBoost)

Univariate feature selection precedes a gradient-boosted tree classifier (`multi:softprob` objective). Class-balanced sample weights, inversely proportional to class frequency, are applied during fitting, and 60 randomly sampled configurations are evaluated by macro-averaged F1 in 5-fold cross-validation. Note that `scipy.stats.uniform(loc, scale)` samples from the interval [`loc`, `loc + scale`].

In [ ]:
# Class-balanced sample weights: n_samples / (n_classes * n_samples_in_class)
xgb_sample_weight = compute_sample_weight("balanced", y_train)

xgb_pipeline = Pipeline([
    ("sel", SelectKBest(score_func=f_classif)),
    ("clf", XGBClassifier(
        objective="multi:softprob",
        num_class=len(np.unique(y_train)),
        tree_method="hist",
        n_jobs=-1,
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
    )),
])

xgb_param_dist = {
    "sel__k": [100, 200, 400, "all"],
    "clf__n_estimators": randint(200, 600),
    "clf__max_depth": randint(3, 8),
    "clf__learning_rate": uniform(0.05, 0.3),
    "clf__subsample": uniform(0.6, 0.3),
    "clf__colsample_bytree": uniform(0.4, 0.8),
    "clf__min_child_weight": randint(1, 5),
    "clf__reg_lambda": uniform(0.5, 2.0),
    "clf__reg_alpha": uniform(0.0, 1.0),
}

xgb_search = RandomizedSearchCV(xgb_pipeline, param_distributions=xgb_param_dist, n_iter=60,
                                cv=cv_tuning, scoring="f1_macro", n_jobs=-1,
                                random_state=RANDOM_STATE, verbose=1)
xgb_search.fit(X_train, y_train, clf__sample_weight=xgb_sample_weight)
xgb_best = xgb_search.best_estimator_

print("Best hyperparameters:", xgb_search.best_params_)
print(f"Best cross-validated macro-F1: {xgb_search.best_score_:.4f}")

In [ ]:
evaluate_on_holdout("XGBoost", xgb_best, X_train, y_train, X_test, y_test)

### 4.7 Model 5: Support vector machine (SVM)

A standardisation step precedes an SVM with a polynomial or radial-basis-function kernel. The regularisation parameter, kernel coefficient, polynomial degree and independent term are tuned by exhaustive grid search (540 configurations × 5 folds); `degree` and `coef0` affect only the polynomial kernel.

In [ ]:
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(random_state=RANDOM_STATE)),
])

svm_param_grid = {
    "svc__C": [0.01, 0.1, 1, 10, 100],
    "svc__kernel": ["poly", "rbf"],
    "svc__gamma": ["scale", 1, 0.1, 0.01, 0.001, 0.0001],
    "svc__degree": [2, 3, 4],          # polynomial kernel only
    "svc__coef0": [0.0, 0.1, 1.0],     # polynomial kernel only
}

svm_search = GridSearchCV(svm_pipeline, param_grid=svm_param_grid, cv=cv_tuning,
                          scoring="accuracy", n_jobs=-1, verbose=1)
svm_search.fit(X_train, y_train)
svm_best = svm_search.best_estimator_

print("Best hyperparameters:", svm_search.best_params_)
print(f"Best cross-validated accuracy: {svm_search.best_score_:.4f}")

In [ ]:
evaluate_on_holdout("SVM", svm_best, X_train, y_train, X_test, y_test)

#### Summary of hold-out performance

In [ ]:
holdout_summary = pd.DataFrame(holdout_results).T.round(4)
holdout_summary.index.name = "Model"
save_table(holdout_summary, "holdout_test_summary")
holdout_summary

### 4.8 Model comparison by repeated stratified cross-validation

The five tuned models are compared on the complete data set (training and test partitions combined) using 5-fold stratified cross-validation repeated three times, i.e. 15 folds per model. Because the same splitter seed is used for every model, the per-fold scores are paired. Macro-averaged metrics are reported as they are robust to class imbalance. The two best-ranked models are compared with a paired Wilcoxon signed-rank test on the per-fold macro-F1.

In [ ]:
tuned_models = {
    "KNN": knn_best,
    "Random Forest": rf_best,
    "Logistic Regression": logreg_best,
    "XGBoost": xgb_best,
    "SVM": svm_best,
}

# Feature matrix used by each model (KNN was tuned without the excluded genes)
feature_sets = {name: (X_knn if name == "KNN" else X) for name in tuned_models}

scoring = {
    "F1_macro": "f1_macro",
    "Balanced_acc": "balanced_accuracy",
    "Precision_macro": "precision_macro",
    "Recall_macro": "recall_macro",
}

records, f1_by_model = [], {}
for name, model in tuned_models.items():
    scores = cross_validate(model, feature_sets[name], y, cv=cv_report, scoring=scoring, n_jobs=-1)
    f1_by_model[name] = scores["test_F1_macro"]
    row = {"Model": name}
    for metric in scoring:
        values = scores[f"test_{metric}"]
        row[f"{metric}_mean"], row[f"{metric}_sd"] = values.mean(), values.std()
    records.append(row)

cv_numeric = pd.DataFrame(records).set_index("Model").sort_values("F1_macro_mean", ascending=False)
cv_table = pd.DataFrame({
    metric: cv_numeric[f"{metric}_mean"].map("{:.3f}".format) + " ± " + cv_numeric[f"{metric}_sd"].map("{:.3f}".format)
    for metric in scoring
})
save_table(cv_numeric, "repeated_cv_summary")
print(cv_table.to_string())

# Paired Wilcoxon signed-rank test between the two best models (macro-F1 per fold)
best_two = cv_table.index[:2].tolist()
_, p_value = wilcoxon(f1_by_model[best_two[0]], f1_by_model[best_two[1]])
print(f"\nWilcoxon signed-rank test, {best_two[0]} vs {best_two[1]} (macro-F1 per fold): p = {p_value:.3f}")
if np.isnan(p_value):
    print("The test is undefined: the per-fold scores of the two models are identical.")
elif p_value < 0.05:
    print("Significant difference (p < 0.05).")
else:
    print("No significant difference (p >= 0.05): the two models perform equivalently.")

## 5. External validation on the INC Bogotá cohort

The tuned models are applied to an independent cohort of Luminal B samples. Because only one class is present, multi-class accuracy and F1 are not informative; the interpretable metric is the **sensitivity for Luminal B**, i.e. the fraction of samples assigned to the correct class.

Two sources of distribution shift must be borne in mind when interpreting the results: (i) the external matrix is VST-normalised, whereas the TCGA matrix is expressed as log2(FPKM + 1), and no re-scaling or batch correction is applied; and (ii) model genes absent from the external panel are imputed with a constant value (Section 5.1).

### 5.1 Loading and gene-space alignment

The external matrix is transposed to a samples × genes layout and reindexed onto the gene space of each model. Genes absent from the external cohort are imputed with a single constant, namely the mean of the per-gene means of the genes observed in the external cohort. KNN uses the feature space without the excluded genes.

In [ ]:
inc_raw = pd.read_csv(INC_EXPRESSION_FILE, index_col=0, encoding="utf-8-sig")

# genes x samples -> samples x genes. Rows that are entirely non-numeric
# (metadata rows, if any) are coerced to NaN and discarded.
inc_expression = inc_raw.apply(pd.to_numeric, errors="coerce").dropna(how="all").T
assert inc_expression.columns.is_unique, "Duplicated gene symbols in the external cohort."

n_discarded = len(inc_raw) - inc_expression.shape[1]
print(f"External cohort: {inc_expression.shape[0]} samples x {inc_expression.shape[1]:,} genes "
      f"({n_discarded} non-numeric row(s) discarded)")


def align_to_model_genes(expression, model_genes):
    """Reindex an external expression matrix onto the gene space used for training.

    Model genes absent from the external cohort are imputed with a single
    constant: the mean of the per-gene means of the genes observed in the
    external cohort. Returns the aligned matrix and the list of imputed genes.
    """
    present = [g for g in model_genes if g in expression.columns]
    missing = [g for g in model_genes if g not in expression.columns]
    aligned = expression.reindex(columns=model_genes)
    if missing:
        aligned[missing] = expression[present].mean(axis=0).mean()
    return aligned, missing


X_inc_knn, missing_knn = align_to_model_genes(inc_expression, X_train_knn.columns.tolist())
X_inc_full, missing_full = align_to_model_genes(inc_expression, X_train.columns.tolist())

print(f"KNN feature space : {len(missing_knn):,} imputed genes | remaining NaN: {int(X_inc_knn.isna().sum().sum())}")
print(f"Full feature space: {len(missing_full):,} imputed genes | remaining NaN: {int(X_inc_full.isna().sum().sum())}")

### 5.2 Exploratory comparison with TCGA

Before validation, the expression distributions of the external cohort and of the TCGA Luminal B samples are compared. The comparison is restricted to genes observed in both data sets so that imputed constants do not distort the distributions. Given the different normalisation procedures, differences in scale are expected; they are documented here rather than corrected.

In [ ]:
genes_shared = [g for g in X_train.columns if g in inc_expression.columns]
X_inc_shared = inc_expression[genes_shared]

values = X_inc_shared.to_numpy().ravel()
values = values[~np.isnan(values)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(values, bins=50, color="steelblue", edgecolor="black", linewidth=0.3)
axes[0].set(title="Global expression distribution – INC cohort",
            xlabel="Expression (VST)", ylabel="Frequency")

kde = gaussian_kde(values)
grid = np.linspace(values.min(), values.max(), 300)
axes[1].fill_between(grid, kde(grid), alpha=0.3, color="steelblue")
axes[1].plot(grid, kde(grid), color="steelblue", lw=2)
axes[1].set(title="Density – INC cohort", xlabel="Expression (VST)", ylabel="Density")

plt.tight_layout()
save_figure("inc_expression_distribution")
plt.show()

In [ ]:
X_lumb_tcga = X.loc[y == LABEL_TO_CODE["LumB"], genes_shared]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Per-sample median expression
axes[0].hist(X_lumb_tcga.median(axis=1), bins=30, alpha=0.6, color="orange",
             label="TCGA LumB", edgecolor="black", linewidth=0.3)
axes[0].hist(X_inc_shared.median(axis=1), bins=12, alpha=0.6, color="steelblue",
             label="INC LumB", edgecolor="black", linewidth=0.3)
axes[0].set(title="Per-sample median expression", xlabel="Median")
axes[0].legend()

# Per-gene variance
var_tcga, var_inc = X_lumb_tcga.var(axis=0), X_inc_shared.var(axis=0)
upper = max(var_tcga.max(), var_inc.max())
axes[1].scatter(var_tcga, var_inc, alpha=0.3, s=10, color="steelblue")
axes[1].plot([0, upper], [0, upper], "r--", alpha=0.5, label="identity")
axes[1].set(title=f"Per-gene variance ({len(genes_shared):,} shared genes)",
            xlabel="TCGA LumB", ylabel="INC LumB")
axes[1].legend()

plt.tight_layout()
save_figure("tcga_vs_inc_comparability")
plt.show()

print(f"INC cohort (shared genes) – mean: {values.mean():.3f} | median: {np.median(values):.3f} | "
      f"SD: {values.std():.3f} | range: [{values.min():.2f}, {values.max():.2f}]")

### 5.3 Sensitivity for Luminal B

Each tuned model predicts the subtype of every external sample. Integer predictions are decoded to subtype names through the shared label mapping, and the sensitivity is the fraction of samples assigned to Luminal B.

In [ ]:
inc_feature_sets = {name: (X_inc_knn if name == "KNN" else X_inc_full) for name in tuned_models}

predicted_labels, rows = {}, []
for name, model in tuned_models.items():
    codes = model.predict(inc_feature_sets[name])
    labels = np.array([CODE_TO_LABEL[int(c)] for c in codes])
    predicted_labels[name] = labels
    n_correct = int((labels == "LumB").sum())
    rows.append({"Model": name, "Correct (LumB)": n_correct, "Total": len(labels),
                 "LumB sensitivity": round(n_correct / len(labels), 3)})

sensitivity = pd.DataFrame(rows).set_index("Model")
save_table(sensitivity, "external_validation_sensitivity")
print("Sensitivity for Luminal B – INC Bogotá cohort")
print(sensitivity.to_string())

#### Distribution of predicted subtypes

All external samples are Luminal B; samples assigned to Luminal A or Basal-like therefore constitute misclassifications.

In [ ]:
prediction_counts = (pd.DataFrame({name: pd.Series(labels).value_counts()
                                   for name, labels in predicted_labels.items()})
                     .T.reindex(columns=CLASS_NAMES).fillna(0).astype(int))

ax = prediction_counts.plot(kind="bar", stacked=True, figsize=(9, 5),
                            color=[COLOR_MAP[c] for c in CLASS_NAMES],
                            edgecolor="black", linewidth=0.3)
ax.set_title(f"Predicted subtypes – INC cohort (n = {len(X_inc_full)}, all Luminal B)")
ax.set_ylabel("Number of samples")
ax.set_xlabel("")
ax.legend(title="Predicted subtype", loc="upper left", bbox_to_anchor=(1.01, 1.0))
plt.xticks(rotation=0)
plt.tight_layout()
save_figure("inc_predicted_subtypes")
plt.show()

## 6. Unsupervised analysis

The structure of the data is explored without using subtype labels, by means of dimensionality reduction (PCA, UMAP) and clustering (*k*-means, hierarchical). Each clustering is evaluated with two comparable metrics: the **silhouette coefficient** (internal cohesion and separation of the clusters) and the **Adjusted Rand Index (ARI)** with respect to the reference subtypes (external concordance).

### 6.1 Preparation

The complete expression matrix (all samples; missing values, if any, replaced by the gene mean) is standardised to zero mean and unit variance per gene, as required by PCA, distance-based clustering and UMAP. Subtype labels are not used to fit any unsupervised model; they serve only to colour the projections and to compute the ARI.

In [ ]:
X_imputed = X.fillna(X.mean())
X_scaled = StandardScaler().fit_transform(X_imputed)

labels_true = merged["Subtype"].to_numpy()
y_true = y.to_numpy()

print(f"Matrix: {X_scaled.shape[0]} samples x {X_scaled.shape[1]:,} genes")

### 6.2 Principal component analysis (PCA)

The explained-variance profile is inspected first, followed by a two-dimensional projection coloured by the reference subtype. The genes with the largest loadings on PC1 are exported as candidate input for functional-enrichment analysis.

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE).fit(X_scaled)
var_ratio = pca_full.explained_variance_ratio_
cum_var = var_ratio.cumsum()
n_pcs_90 = int(np.argmax(cum_var >= 0.90) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(range(1, 21), var_ratio[:20] * 100, "o-")
axes[0].set(title="Variance explained per component", xlabel="Principal component", ylabel="Variance (%)")
axes[1].plot(range(1, len(cum_var) + 1), cum_var * 100, "-")
axes[1].axhline(90, color="r", ls="--", lw=0.8)
axes[1].axvline(n_pcs_90, color="r", ls="--", lw=0.8)
axes[1].set(title=f"Cumulative variance (90% reached with {n_pcs_90} PCs)",
            xlabel="Number of components", ylabel="Cumulative variance (%)")
plt.tight_layout()
save_figure("pca_explained_variance")
plt.show()

print(f"PC1: {var_ratio[0] * 100:.2f}% | PC2: {var_ratio[1] * 100:.2f}% | 90% of the variance with {n_pcs_90} components")

In [ ]:
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca_2d.fit_transform(X_scaled)
pc_var = pca_2d.explained_variance_ratio_ * 100

plt.figure(figsize=(8, 6))
for subtype, color in COLOR_MAP.items():
    mask = labels_true == subtype
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=f"{subtype} (n={mask.sum()})", alpha=0.6, s=25)
plt.xlabel(f"PC1 ({pc_var[0]:.1f}%)")
plt.ylabel(f"PC2 ({pc_var[1]:.1f}%)")
plt.title("PCA – reference subtypes")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
save_figure("pca_projection")
plt.show()

In [ ]:
loadings = pd.DataFrame(pca_full.components_[:2].T, index=X_imputed.columns, columns=["PC1", "PC2"])
top_pc1_positive = loadings["PC1"].sort_values(ascending=False).head(10)
top_pc1_negative = loadings["PC1"].sort_values().head(10)

print("Top 10 genes by positive PC1 loading:")
print(top_pc1_positive.to_string())
print("\nTop 10 genes by negative PC1 loading:")
print(top_pc1_negative.to_string())

save_table(top_pc1_positive.rename("PC1_loading"), "top10_pc1_positive_genes", DAVID_DIR)
save_table(top_pc1_negative.rename("PC1_loading"), "top10_pc1_negative_genes", DAVID_DIR)

### 6.3 K-means clustering

The number of clusters is inspected with the elbow criterion (inertia) and the mean silhouette coefficient for *k* = 2, …, 7. The final model uses *k* = 3, matching the number of reference subtypes.

In [ ]:
k_range = range(2, 8)
inertias, silhouettes = [], []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(k_range, inertias, "o-")
axes[0].set(title="Elbow method", xlabel="k", ylabel="Inertia")
axes[1].plot(k_range, silhouettes, "o-", color="crimson")
axes[1].set(title="Mean silhouette coefficient vs k", xlabel="k", ylabel="Silhouette")
plt.tight_layout()
save_figure("kmeans_k_selection")
plt.show()

# Final model with k = 3 (number of reference subtypes)
km_labels = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10).fit_predict(X_scaled)

### 6.4 Hierarchical clustering

Agglomerative clustering with Ward linkage is applied to the standardised data. The dendrogram is truncated to the last 12 merges for legibility, and the tree is cut at three clusters.

In [ ]:
linkage_matrix = linkage(X_scaled, method="ward")

plt.figure(figsize=(12, 5))
dendrogram(linkage_matrix, truncate_mode="lastp", p=12, leaf_rotation=90.0, show_contracted=True)
plt.title("Dendrogram (Ward linkage, truncated)")
plt.xlabel("Samples")
plt.ylabel("Distance")
plt.tight_layout()
save_figure("hierarchical_dendrogram")
plt.show()

# Cut the tree into three clusters
hc_labels = fcluster(linkage_matrix, 3, criterion="maxclust")

### 6.5 UMAP

A two-dimensional UMAP embedding of the standardised data is computed and coloured by reference subtype. *k*-means (*k* = 3) is then applied in the embedding space so that this approach can be compared with the other clusterings (the silhouette coefficient is nevertheless computed in the standardised gene space, see Section 6.6).

In [ ]:
reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=RANDOM_STATE)
embedding = reducer.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
for subtype, color in COLOR_MAP.items():
    mask = labels_true == subtype
    plt.scatter(embedding[mask, 0], embedding[mask, 1], c=color, label=f"{subtype} (n={mask.sum()})", alpha=0.6, s=25)
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.title("UMAP – reference subtypes")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
save_figure("umap_projection")
plt.show()

# k-means in the UMAP space
umap_labels = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10).fit_predict(embedding)

### 6.6 Comparison of clustering solutions

The silhouette coefficient (internal validity) and the ARI with respect to the reference subtypes (external validity) are computed for every clustering. For comparability, the silhouette is evaluated in the standardised gene space for all methods.

In [ ]:
def evaluate_clustering(name, labels):
    """Return the silhouette coefficient and the ARI vs the reference subtypes."""
    return {
        "Method": name,
        "Silhouette": round(silhouette_score(X_scaled, labels), 3),
        "ARI_vs_subtype": round(adjusted_rand_score(y_true, labels), 3),
    }


clustering_metrics = pd.DataFrame([
    evaluate_clustering("K-means", km_labels),
    evaluate_clustering("Hierarchical (Ward)", hc_labels),
    evaluate_clustering("UMAP + K-means", umap_labels),
]).set_index("Method")
save_table(clustering_metrics, "unsupervised_clustering_metrics")

print(clustering_metrics.to_string())
print(f"\nPCA:  PC1 + PC2 explain {(var_ratio[0] + var_ratio[1]) * 100:.1f}% of the variance")
print("UMAP: 2-D projection (n_neighbors = 30, min_dist = 0.1)")

In [ ]:
# Each K-means cluster is labelled with its majority reference subtype (for colouring only)
cluster_to_subtype = (pd.DataFrame({"cluster": km_labels, "subtype": labels_true})
                      .groupby("cluster")["subtype"]
                      .agg(lambda s: s.value_counts().index[0])
                      .to_dict())
ari_km = adjusted_rand_score(y_true, km_labels)
sil_km = silhouette_score(X_scaled, km_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for subtype, color in COLOR_MAP.items():
    mask = labels_true == subtype
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=subtype, alpha=0.6, s=20)
axes[0].set_title(f"Reference subtypes\nPC1 = {pc_var[0]:.1f}% | PC2 = {pc_var[1]:.1f}%")
axes[0].legend()

for cluster in range(3):
    mask = km_labels == cluster
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c=COLOR_MAP[cluster_to_subtype[cluster]], alpha=0.6, s=20,
                    label=f"Cluster {cluster} → {cluster_to_subtype[cluster]} (n={mask.sum()})")
axes[1].set_title(f"K-means clusters (k = 3)\nARI = {ari_km:.3f} | Silhouette = {sil_km:.3f}")
axes[1].legend()

for ax in axes:
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
plt.tight_layout()
save_figure("reference_subtypes_vs_kmeans")
plt.show()

## 7. Summary and limitations

**Workflow.** Five classifiers were tuned on a predefined training partition, assessed on a held-out test partition and compared on the complete data set by repeated stratified cross-validation; the tuned models were then applied to an independent external cohort, and the data structure was characterised by unsupervised methods. All summary tables are written to `results/`, all figures to `figures/` and the PC1 gene lists to `results/`.

**Limitations and considerations for interpretation.**

1. *Optimism of the repeated cross-validation.* The comparison in Section 4.8 is performed on the complete data set with hyperparameters that were selected on the training partition, which slightly favours all models. A nested cross-validation would provide an unbiased estimate of generalisation performance.
2. *Dependence between folds.* Cross-validation folds share training data, so the per-fold scores are not independent and the Wilcoxon *p*-value should be regarded as approximate.
3. *Class weights in the model comparison.* The sample weights used to tune XGBoost are not propagated to the repeated cross-validation of Section 4.8; the comparison therefore refits XGBoost without class weighting.
4. *External validation.* The external cohort contains a single class, is normalised differently from the training data (VST vs log2(FPKM + 1)) and lacks part of the model gene space, which is imputed with a constant. The reported sensitivity should therefore be interpreted as a robustness check under distribution shift rather than as an estimate of the expected clinical accuracy.
5. *Unsupervised analyses.* Clustering and projections use all samples (training and test partitions combined). Because subtype labels are not used to fit these models, this does not constitute label leakage, but the resulting metrics are descriptive and are not comparable with the supervised test-set performance. Silhouette coefficients in a high-dimensional standardised space are expected to be low.